In [ ]:
import os
import json
import csv
from dataclasses import dataclass, asdict
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import torch
from torch import nn
from tqdm import tqdm

import datasets
import unet
from SplitNet import SplitNet


# -----------------------------------------------------------------------------
# Setup
# -----------------------------------------------------------------------------

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

EVAL_DIR = "./results_eval"
os.makedirs(EVAL_DIR, exist_ok=True)


# -----------------------------------------------------------------------------
# Config
# -----------------------------------------------------------------------------


@dataclass
class EvalConfig:
    model_path: str

    # Optional. If not provided, this file will try to infer these from the
    # saved *_summary.json created by experiment_runner.py.
    model_type: Optional[str] = None
    dataset_mode: Optional[str] = None
    training_mode: Optional[str] = None
    channels: str = "KP"

    # Test set / evaluation dataset settings
    test_sims_path: str = "../test_sims.npy"
    sim_max: Optional[int] = 250
    steps: Tuple[int, int] = (0, 200)
    types: Tuple[int, ...] = (0,)
    points_per_side: int = 3
    radius: int = 5

    # Evaluation behavior
    batch_size: int = 1
    eval_dir: str = EVAL_DIR
    run_name: Optional[str] = None

    # For random-trained models, there is no RandomDenseDatasetFull in datasets.py.
    # This controls what full-target test dataset to use instead.
    # Usually "fixed" is the safest standardized choice.
    random_eval_as: str = "fixed"


# -----------------------------------------------------------------------------
# Small utilities
# -----------------------------------------------------------------------------


def model_name_from_path(path: str) -> str:
    name = os.path.basename(path)
    for suffix in ["_best_state.pt", "_final_state.pt", ".pt", ".pth"]:
        if name.endswith(suffix):
            return name[: -len(suffix)]
    return os.path.splitext(name)[0]


def safe_float(x):
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().item())
    if isinstance(x, np.generic):
        return float(x)
    return x


def to_numpy_dict(d: Dict[str, torch.Tensor]) -> Dict[str, np.ndarray]:
    out = {}
    for k, v in d.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.detach().cpu().numpy()
        else:
            out[k] = np.asarray(v)
    return out


def find_summary_for_model(model_path: str) -> Optional[str]:
    """
    If model_path is something like:
        fixed_physics_limited_splitnet_attn_darcy_1_best_state.pt
    this tries to find:
        fixed_physics_limited_splitnet_attn_darcy_1_summary.json
    """
    folder = os.path.dirname(model_path)
    base = os.path.basename(model_path)

    for suffix in ["_best_state.pt", "_final_state.pt"]:
        if base.endswith(suffix):
            candidate = os.path.join(folder, base[: -len(suffix)] + "_summary.json")
            if os.path.exists(candidate):
                return candidate

    candidate = os.path.join(folder, model_name_from_path(model_path) + "_summary.json")
    if os.path.exists(candidate):
        return candidate

    return None


def load_json(path: str) -> Dict[str, Any]:
    with open(path, "r") as f:
        return json.load(f)


def fill_config_from_summary(config: EvalConfig) -> EvalConfig:
    """
    Pulls model_type, dataset_mode, training_mode, and channels from the summary
    JSON saved by experiment_runner.py, when available.
    """
    summary_path = find_summary_for_model(config.model_path)
    if summary_path is None:
        return config

    summary = load_json(summary_path)
    saved_config = summary.get("config", {})

    if config.model_type is None:
        config.model_type = saved_config.get("model_type") or summary.get("model_type")
    if config.dataset_mode is None:
        config.dataset_mode = saved_config.get("dataset_mode") or summary.get("dataset_mode")
    if config.training_mode is None:
        config.training_mode = saved_config.get("training_mode") or summary.get("training_mode")

    config.channels = saved_config.get("channels", config.channels)

    return config


# -----------------------------------------------------------------------------
# Model loading
# -----------------------------------------------------------------------------


def make_model(model_type: str, channels: str = "KP") -> nn.Module:
    model_type = model_type.lower()

    if channels == "all":
        num_channels = 3
    elif channels == "KP":
        num_channels = 2
    elif channels in ["K", "P", "phi"]:
        num_channels = 1
    else:
        raise ValueError("channels must be 'all', 'KP', 'K', 'P', or 'phi'.")

    if model_type == "splitnet_attn":
        return SplitNet(attn=True).to(DEVICE)
    if model_type == "splitnet":
        return SplitNet(attn=False).to(DEVICE)
    if model_type == "unet":
        return unet.SmallUnet(channels=num_channels).to(DEVICE)
    if model_type == "attn_unet":
        return unet.AttnUnet(channels=num_channels).to(DEVICE)

    raise ValueError(f"Unknown model_type: {model_type}")


def load_model(config: EvalConfig) -> nn.Module:
    config = fill_config_from_summary(config)

    if config.model_type is None:
        raise ValueError(
            "Could not infer model_type. Pass model_type='splitnet_attn', 'splitnet', 'unet', or 'attn_unet'."
        )

    loaded = torch.load(config.model_path, map_location=DEVICE, weights_only=False)

    # Preferred case: saved state_dict from experiment_runner.py
    if isinstance(loaded, dict):
        model = make_model(config.model_type, config.channels)
        model.load_state_dict(loaded)
    else:
        # Older case: whole model object was saved.
        model = loaded.to(DEVICE)

    model.eval()
    return model


# -----------------------------------------------------------------------------
# Darcy / metric functions
# -----------------------------------------------------------------------------


def darcy_residual_map(out: torch.Tensor) -> torch.Tensor:
    """
    Computes div(K * grad(P)). Output channel order must be K, P, ...
    Returns [B, 1, H, W].
    """
    if out.shape[1] < 2:
        raise ValueError("Darcy residual requires at least 2 channels: K and P.")

    k = out[:, 0:1]
    p = out[:, 1:2]

    p_y, p_x = torch.gradient(p, dim=(-2, -1))
    flux_y = k * p_y
    flux_x = k * p_x

    div_y = torch.gradient(flux_y, spacing=(1,), dim=(-2,))[0]
    div_x = torch.gradient(flux_x, spacing=(1,), dim=(-1,))[0]

    return div_y + div_x


def darcy_loss_from_output(out: torch.Tensor) -> torch.Tensor:
    return (darcy_residual_map(out) ** 2).mean()


def expand_mask(mask: torch.Tensor, channels: int) -> torch.Tensor:
    if mask.dim() == 2:
        mask = mask.unsqueeze(0).unsqueeze(0)
    elif mask.dim() == 3:
        mask = mask.unsqueeze(1)
    return mask.expand(-1, channels, -1, -1)


def mse_nmse_energy(pred: torch.Tensor, target: torch.Tensor, mask: Optional[torch.Tensor] = None):
    """
    Returns mean NMSE, MSE, and target energy over a batch.
    Works for full image or masked region.
    """
    if pred.shape != target.shape:
        raise ValueError(f"Shape mismatch: {pred.shape} vs {target.shape}")

    if mask is not None:
        mask = expand_mask(mask.bool().to(pred.device), pred.shape[1]).float()
        err = ((pred - target) ** 2) * mask
        energy_map = (target ** 2) * mask
        denom = mask.reshape(pred.shape[0], -1).sum(dim=1).clamp_min(1e-8)
        mse = err.reshape(pred.shape[0], -1).sum(dim=1) / denom
        energy = energy_map.reshape(pred.shape[0], -1).sum(dim=1) / denom
    else:
        mse = ((pred - target) ** 2).reshape(pred.shape[0], -1).mean(dim=1)
        energy = (target ** 2).reshape(pred.shape[0], -1).mean(dim=1)

    nmse = mse / energy.clamp_min(1e-8)
    return nmse.mean(), mse.mean(), energy.mean()


def mae_metric(pred: torch.Tensor, target: torch.Tensor, mask: Optional[torch.Tensor] = None):
    if mask is not None:
        mask = expand_mask(mask.bool().to(pred.device), pred.shape[1]).float()
        err = (pred - target).abs() * mask
        denom = mask.reshape(pred.shape[0], -1).sum(dim=1).clamp_min(1e-8)
        return (err.reshape(pred.shape[0], -1).sum(dim=1) / denom).mean()
    return (pred - target).abs().mean()


def bias_metric(pred: torch.Tensor, target: torch.Tensor, mask: Optional[torch.Tensor] = None):
    if mask is not None:
        mask = expand_mask(mask.bool().to(pred.device), pred.shape[1]).float()
        diff = (pred - target) * mask
        denom = mask.reshape(pred.shape[0], -1).sum(dim=1).clamp_min(1e-8)
        return (diff.reshape(pred.shape[0], -1).sum(dim=1) / denom).mean()
    return (pred - target).mean()


def align_channels(label: torch.Tensor, out: torch.Tensor) -> torch.Tensor:
    if label.shape[1] == out.shape[1]:
        return label
    if label.shape[1] > out.shape[1]:
        return label[:, : out.shape[1]]
    raise ValueError(f"Label has {label.shape[1]} channels but output has {out.shape[1]} channels.")


# -----------------------------------------------------------------------------
# Dataset / mask helpers
# -----------------------------------------------------------------------------


def load_test_sims(path: str, sim_max: Optional[int]) -> np.ndarray:
    sims = np.load(path)
    if sim_max is not None:
        sims = sims[sims < sim_max]
    return sims


def build_eval_dataset(config: EvalConfig):
    """
    Evaluation should use full-target dense datasets so that full image,
    observed-region, and unobserved-region errors are all meaningful.

    For random-trained models, your datasets.py does not include a RandomDenseDatasetFull.
    By default we evaluate those on FixedDenseDatasetFull as a standardized test set.
    """
    config = fill_config_from_summary(config)

    if config.dataset_mode is None:
        raise ValueError("Could not infer dataset_mode. Pass dataset_mode='fixed', 'border', or 'random'.")

    sims = load_test_sims(config.test_sims_path, config.sim_max)
    kwargs = dict(
        points_per_side=config.points_per_side,
        radius=config.radius,
        steps=config.steps,
        types=list(config.types),
        channels=config.channels,
    )

    eval_mode = config.dataset_mode
    if eval_mode == "random":
        eval_mode = config.random_eval_as

    if eval_mode == "fixed":
        return datasets.FixedDenseDatasetFull(sims, **kwargs), "fixed", sims
    if eval_mode == "border":
        return datasets.BorderDenseDatasetFull(sims, **kwargs), "border", sims

    raise ValueError("Evaluation dataset must be 'fixed' or 'border'. For random, set random_eval_as='fixed' or 'border'.")


def observed_mask_from_feature(feat: torch.Tensor) -> torch.Tensor:
    """
    Infers observed/revealed region from dataset input sample.

    This avoids rebuilding masks manually and keeps evaluation tied to whatever
    the dataset actually returns. It is not perfect if an observed value is
    exactly zero after normalization, but for region-level evaluation this is
    usually acceptable. For fixed/border masks, the region is large enough that
    this is stable.

    feat shape: [B, C, H, W]
    returns: [B, H, W] bool
    """
    return feat.abs().sum(dim=1) > 0


# -----------------------------------------------------------------------------
# Main evaluation: slow step
# -----------------------------------------------------------------------------


def evaluate_model_to_npz(config: EvalConfig) -> str:
    """
    Slow step.

    Loads one trained model, evaluates it over the test dataset, and saves a .npz
    with metric matrices over:
        type x sim x step

    Saved arrays include:
        mse/nmse/energy total
        mse/nmse/energy per channel K/P/phi if available
        mse/nmse in observed mask region
        mse/nmse outside observed mask region
        Darcy loss over full output
        MAE and bias metrics
    """
    config = fill_config_from_summary(config)
    model = load_model(config)
    dataset, eval_mask_mode, sims = build_eval_dataset(config)

    run_name = config.run_name or model_name_from_path(config.model_path)
    out_path = os.path.join(config.eval_dir, f"eval_{run_name}.npz")
    os.makedirs(config.eval_dir, exist_ok=True)

    K_types = len(dataset.types)
    n_sims = dataset.sims.shape[0]
    T = dataset.num_steps()

    shape = (K_types, n_sims, T)

    metrics = {
        "darcy": torch.zeros(shape, dtype=torch.float32),
        "nmse_total": torch.zeros(shape, dtype=torch.float32),
        "mse_total": torch.zeros(shape, dtype=torch.float32),
        "energy_total": torch.zeros(shape, dtype=torch.float32),
        "mae_total": torch.zeros(shape, dtype=torch.float32),
        "bias_total": torch.zeros(shape, dtype=torch.float32),
        "nmse_inmask": torch.zeros(shape, dtype=torch.float32),
        "mse_inmask": torch.zeros(shape, dtype=torch.float32),
        "mae_inmask": torch.zeros(shape, dtype=torch.float32),
        "bias_inmask": torch.zeros(shape, dtype=torch.float32),
        "nmse_outmask": torch.zeros(shape, dtype=torch.float32),
        "mse_outmask": torch.zeros(shape, dtype=torch.float32),
        "mae_outmask": torch.zeros(shape, dtype=torch.float32),
        "bias_outmask": torch.zeros(shape, dtype=torch.float32),
        "nmse_k": torch.zeros(shape, dtype=torch.float32),
        "mse_k": torch.zeros(shape, dtype=torch.float32),
        "energy_k": torch.zeros(shape, dtype=torch.float32),
        "mae_k": torch.zeros(shape, dtype=torch.float32),
        "bias_k": torch.zeros(shape, dtype=torch.float32),
        "nmse_p": torch.zeros(shape, dtype=torch.float32),
        "mse_p": torch.zeros(shape, dtype=torch.float32),
        "energy_p": torch.zeros(shape, dtype=torch.float32),
        "mae_p": torch.zeros(shape, dtype=torch.float32),
        "bias_p": torch.zeros(shape, dtype=torch.float32),
    }

    # Only filled if output has 3 channels.
    phi_metrics = {
        "nmse_phi": torch.zeros(shape, dtype=torch.float32),
        "mse_phi": torch.zeros(shape, dtype=torch.float32),
        "energy_phi": torch.zeros(shape, dtype=torch.float32),
        "mae_phi": torch.zeros(shape, dtype=torch.float32),
        "bias_phi": torch.zeros(shape, dtype=torch.float32),
    }
    has_phi = False

    print(f"\n=== Evaluating {run_name} ===")
    print(f"model_path: {config.model_path}")
    print(f"model_type: {config.model_type}, trained_dataset_mode: {config.dataset_mode}, eval_mask_mode: {eval_mask_mode}")
    print(f"shape: types={K_types}, sims={n_sims}, steps={T}")

    model.eval()

    with torch.inference_mode():
        for t in tqdm(range(K_types), desc="types"):
            for s in tqdm(range(n_sims), desc="sims", leave=False):
                for step in range(T):
                    idx = t * n_sims * T + s * T + step
                    feat, label = dataset[idx]

                    feat = feat.unsqueeze(0).to(DEVICE)
                    label = label.unsqueeze(0).to(DEVICE)

                    out = model(feat)
                    label = align_channels(label, out)

                    obs_mask = observed_mask_from_feature(feat).to(DEVICE)
                    out_mask = ~obs_mask

                    darcy = darcy_loss_from_output(out)
                    nmse, mse, energy = mse_nmse_energy(out, label)

                    metrics["darcy"][t, s, step] = safe_float(darcy)
                    metrics["nmse_total"][t, s, step] = safe_float(nmse)
                    metrics["mse_total"][t, s, step] = safe_float(mse)
                    metrics["energy_total"][t, s, step] = safe_float(energy)
                    metrics["mae_total"][t, s, step] = safe_float(mae_metric(out, label))
                    metrics["bias_total"][t, s, step] = safe_float(bias_metric(out, label))

                    nmse_in, mse_in, _ = mse_nmse_energy(out, label, obs_mask)
                    nmse_out, mse_out, _ = mse_nmse_energy(out, label, out_mask)

                    metrics["nmse_inmask"][t, s, step] = safe_float(nmse_in)
                    metrics["mse_inmask"][t, s, step] = safe_float(mse_in)
                    metrics["mae_inmask"][t, s, step] = safe_float(mae_metric(out, label, obs_mask))
                    metrics["bias_inmask"][t, s, step] = safe_float(bias_metric(out, label, obs_mask))

                    metrics["nmse_outmask"][t, s, step] = safe_float(nmse_out)
                    metrics["mse_outmask"][t, s, step] = safe_float(mse_out)
                    metrics["mae_outmask"][t, s, step] = safe_float(mae_metric(out, label, out_mask))
                    metrics["bias_outmask"][t, s, step] = safe_float(bias_metric(out, label, out_mask))

                    # Per-channel metrics. K=0, P=1, phi=2 if present.
                    for channel_name, ch in [("k", 0), ("p", 1)]:
                        if out.shape[1] > ch:
                            pred_ch = out[:, ch : ch + 1]
                            label_ch = label[:, ch : ch + 1]
                            nmse_c, mse_c, energy_c = mse_nmse_energy(pred_ch, label_ch)
                            metrics[f"nmse_{channel_name}"][t, s, step] = safe_float(nmse_c)
                            metrics[f"mse_{channel_name}"][t, s, step] = safe_float(mse_c)
                            metrics[f"energy_{channel_name}"][t, s, step] = safe_float(energy_c)
                            metrics[f"mae_{channel_name}"][t, s, step] = safe_float(mae_metric(pred_ch, label_ch))
                            metrics[f"bias_{channel_name}"][t, s, step] = safe_float(bias_metric(pred_ch, label_ch))

                    if out.shape[1] >= 3:
                        has_phi = True
                        pred_phi = out[:, 2:3]
                        label_phi = label[:, 2:3]
                        nmse_phi, mse_phi, energy_phi = mse_nmse_energy(pred_phi, label_phi)
                        phi_metrics["nmse_phi"][t, s, step] = safe_float(nmse_phi)
                        phi_metrics["mse_phi"][t, s, step] = safe_float(mse_phi)
                        phi_metrics["energy_phi"][t, s, step] = safe_float(energy_phi)
                        phi_metrics["mae_phi"][t, s, step] = safe_float(mae_metric(pred_phi, label_phi))
                        phi_metrics["bias_phi"][t, s, step] = safe_float(bias_metric(pred_phi, label_phi))

    if has_phi:
        metrics.update(phi_metrics)

    arrays = to_numpy_dict(metrics)
    arrays["sims"] = np.asarray(sims)
    arrays["steps"] = np.arange(config.steps[0], config.steps[1])
    arrays["types"] = np.asarray(config.types)

    np.savez(out_path, **arrays)

    # Save metadata next to npz.
    meta_path = out_path.replace(".npz", "_meta.json")
    save_meta = asdict(config)
    save_meta["eval_mask_mode"] = eval_mask_mode
    save_meta["output_path"] = out_path
    with open(meta_path, "w") as f:
        json.dump(save_meta, f, indent=2)

    # Save compact summary CSV/JSON for quick comparison.
    save_eval_summary(out_path)

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"Saved eval arrays: {out_path}")
    print(f"Saved metadata:    {meta_path}")
    return out_path


# -----------------------------------------------------------------------------
# Summary / comparison: fast step
# -----------------------------------------------------------------------------


def load_eval_npz(path: str) -> Dict[str, np.ndarray]:
    data = np.load(path)
    return {k: data[k] for k in data.files}


def summarize_eval_arrays(arrays: Dict[str, np.ndarray]) -> Dict[str, float]:
    """
    Produces one-row aggregate metrics from an eval .npz.
    """
    keys = [
        "darcy",
        "nmse_total", "mse_total", "mae_total", "bias_total",
        "nmse_inmask", "mse_inmask", "mae_inmask", "bias_inmask",
        "nmse_outmask", "mse_outmask", "mae_outmask", "bias_outmask",
        "nmse_k", "mse_k", "mae_k", "bias_k",
        "nmse_p", "mse_p", "mae_p", "bias_p",
        "nmse_phi", "mse_phi", "mae_phi", "bias_phi",
    ]

    summary = {}
    for key in keys:
        if key in arrays:
            values = arrays[key].astype(np.float64)
            summary[f"mean_{key}"] = float(np.nanmean(values))
            summary[f"std_{key}"] = float(np.nanstd(values))
            summary[f"median_{key}"] = float(np.nanmedian(values))

    return summary


def save_eval_summary(npz_path: str) -> Dict[str, float]:
    arrays = load_eval_npz(npz_path)
    summary = summarize_eval_arrays(arrays)

    base = npz_path.replace(".npz", "")
    json_path = f"{base}_summary.json"
    csv_path = f"{base}_summary.csv"

    with open(json_path, "w") as f:
        json.dump(summary, f, indent=2)

    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        for k, v in summary.items():
            writer.writerow([k, v])

    return summary


def compare_eval_files(npz_paths: List[str], out_csv: str = "./results_eval/model_comparison.csv") -> str:
    """
    Fast step.

    Reads already-computed .npz eval files and creates one comparison CSV.
    This is what you use after the slow evaluation step has already run.
    """
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)

    rows = []
    all_keys = set()

    for path in npz_paths:
        arrays = load_eval_npz(path)
        summary = summarize_eval_arrays(arrays)
        name = os.path.basename(path).replace("eval_", "").replace(".npz", "")

        row = {"name": name, "npz_path": path}

        meta_path = path.replace(".npz", "_meta.json")
        if os.path.exists(meta_path):
            meta = load_json(meta_path)
            row.update({
                "model_path": meta.get("model_path"),
                "model_type": meta.get("model_type"),
                "dataset_mode": meta.get("dataset_mode"),
                "training_mode": meta.get("training_mode"),
                "channels": meta.get("channels"),
                "eval_mask_mode": meta.get("eval_mask_mode"),
            })

        row.update(summary)
        all_keys.update(row.keys())
        rows.append(row)

    preferred = [
        "name", "model_type", "dataset_mode", "training_mode", "channels", "eval_mask_mode", "model_path", "npz_path",
        "mean_mse_total", "mean_nmse_total", "mean_darcy",
        "mean_mse_inmask", "mean_mse_outmask",
        "mean_mse_k", "mean_mse_p", "mean_mse_phi",
        "mean_mae_total", "mean_bias_total",
    ]
    fieldnames = preferred + sorted(k for k in all_keys if k not in preferred)

    with open(out_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)

    print(f"Saved comparison CSV: {out_csv}")
    return out_csv


# -----------------------------------------------------------------------------
# Convenience helpers
# -----------------------------------------------------------------------------


def evaluate_many(model_paths: List[str], eval_dir: str = EVAL_DIR, **kwargs) -> List[str]:
    """
    Runs the slow eval step for many models.

    Example:
        paths = evaluate_many([
            "minimum_info/final_grid/fixed/fixed_physics_limited_splitnet_attn_darcy_100p0_best_state.pt",
            "minimum_info/final_grid/fixed/fixed_baseline_full_splitnet_attn_nodarcy_best_state.pt",
        ])
    """
    out_paths = []
    for model_path in model_paths:
        cfg = EvalConfig(model_path=model_path, eval_dir=eval_dir, **kwargs)
        out_paths.append(evaluate_model_to_npz(cfg))
    return out_paths


def find_model_states(root: str, kind: str = "best") -> List[str]:
    """
    Finds saved model weight files under a directory.

    kind='best'  -> *_best_state.pt
    kind='final' -> *_final_state.pt
    kind='all'   -> both
    """
    matches = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if kind == "best" and fn.endswith("_best_state.pt"):
                matches.append(os.path.join(dirpath, fn))
            elif kind == "final" and fn.endswith("_final_state.pt"):
                matches.append(os.path.join(dirpath, fn))
            elif kind == "all" and (fn.endswith("_best_state.pt") or fn.endswith("_final_state.pt")):
                matches.append(os.path.join(dirpath, fn))
    return sorted(matches)


# -------------------------------------------------------------------------
# STEP 1: Slow evaluation step.
# -------------------------------------------------------------------------
# This reads model weights and creates saved .npz metric files.

# Example A: evaluate one model
# evaluate_model_to_npz(EvalConfig(
#     model_path="recent_analysis/physics_tests/fixed/fixed_physics_limited_splitnet_attn_darcy_0p0_best_state.pt"
# ))






# Example B: evaluate every best model in a folder
# model_paths = find_model_states("minimum_info/final_grid", kind="best")
# eval_paths = evaluate_many(model_paths, eval_dir="./results_eval")

# -------------------------------------------------------------------------
# STEP 2: Fast comparison step.
# -------------------------------------------------------------------------
# This compares already-saved .npz files without rerunning models.

# eval_paths = [
#     os.path.join("./results_eval", f)
#     for f in os.listdir("./results_eval")
#     if f.endswith(".npz")
# ]
# compare_eval_files(eval_paths, out_csv="./recent_analysis/model_comparison.csv")
pass


In [ ]:
eval_paths = [
    os.path.join("./results_eval", f)
    for f in os.listdir("./results_eval")
    if f.endswith(".npz")
]
compare_eval_files(eval_paths, out_csv="./recent_analysis/model_comparison.csv")

Saved comparison CSV: ./results_eval/model_comparison.csv


'./results_eval/model_comparison.csv'